<a href="https://colab.research.google.com/github/window7458/jiyoongrammar/blob/main/assets/Applio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Applio**
A simple, high-quality voice conversion tool focused on ease of use and performance.

[Support](https://discord.gg/wY7gmqTyEV) — [GitHub](https://github.com/IAHispano/Applio) — [Terms of Use](https://github.com/IAHispano/Applio/blob/main/TERMS_OF_USE.md)

<br>

---

<br>

#### **Acknowledgments**

To all external collaborators for their special help in the following areas:
* Hina (Encryption method)
* Poopmaster (Extra section)
* Shirou (UV installer)
* Bruno5430 (AutoBackup code and general notebook maintenance)

#### **Disclaimer**
By using Applio, you agree to comply with ethical and legal standards, respect intellectual property and privacy rights, avoid harmful or prohibited uses, and accept full responsibility for any outcomes, while Applio disclaims liability and reserves the right to amend these terms.

### **Install Applio**
If the runtime restarts, re-run the installation steps.

In [ ]:
# @title Mount Drive
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  print("❌ Failed to mount drive")

In [ ]:
# @title Setup runtime environment
from IPython.display import clear_output
import codecs

encoded_url = "uggcf://tvguho.pbz/VNUvfcnab/Nccyvb/"
decoded_url = codecs.decode(encoded_url, "rot_13")

repo_name_encoded = "Nccyvb"
repo_name = codecs.decode(repo_name_encoded, "rot_13")

LOGS_PATH = f"/content/{repo_name}/logs"
BACKUPS_PATH = f"/content/drive/MyDrive/{repo_name}Backup"

%cd /content
!git config --global advice.detachedHead false
!git clone {decoded_url} --branch 3.6.2 --single-branch
%cd {repo_name}
clear_output()

!apt update -y
!apt install -y portaudio19-dev

print("Installing requirements...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install -q ngrok jupyter-ui-poll
!npm install -g -q localtunnel &> /dev/null

!python core.py "prerequisites" --models "True" --pretraineds_hifigan "True"
print("✅ Finished installing requirements!")


### **Start Applio**

In [ ]:
# @title Sync with Google Drive
# @markdown 💾 Run this cell to automatically Save/Load models from your mounted drive
# @title
# @markdown This will merge and link your `ApplioBackup` folder from gdrive to this notebook
from IPython.display import display, clear_output
from pathlib import Path

non_bak_folders = ["mute", "reference", "zips", "mute_spin", "mute_spin-v2"]
non_bak_path = "/tmp/rvc_logs"


def press_button(button):
  button.disabled = True


def get_date(path: Path):
  from datetime import datetime
  return datetime.fromtimestamp(int(path.stat().st_mtime))


def get_size(path: Path):
  !du -shx --apparent-size "{path}" > /tmp/size.txt
  return open("/tmp/size.txt").readlines().pop(0).split("	")[0] + "B"


def sync_folders(folder: Path, backup: Path):
  from ipywidgets import widgets
  from jupyter_ui_poll import ui_events
  from time import sleep

  local = widgets.VBox([
      widgets.Label(f"Local: {LOGS_PATH.removeprefix('/content/')}/{folder.name}/"),
      widgets.Label(f"Size: {get_size(folder)}"),
      widgets.Label(f"Last modified: {get_date(folder)}")
  ])
  remote = widgets.VBox([
      widgets.Label(f"Remote: {BACKUPS_PATH.removeprefix('/content/')}/{backup.name}/"),
      widgets.Label(f"Size: {get_size(backup)}"),
      widgets.Label(f"Last modified: {get_date(backup)}")
  ])
  separator = widgets.VBox([
      widgets.Label("|||"),
      widgets.Label("|||"),
      widgets.Label("|||")
  ])
  radio = widgets.RadioButtons(
      options=[
          "Save local model to drive",
          "Keep remote model"
      ]
  )
  button = widgets.Button(
      description="Sync",
      icon="upload",
      tooltip="Sync model"
  )
  button.on_click(press_button)

  clear_output()
  print(f"Your local model '{folder.name}' is in conflict with it's copy in Google Drive.")
  print("Please select which one you want to keep:")
  display(widgets.Box([local, separator, remote]))
  display(radio)
  display(button)

  with ui_events() as poll:
    while not button.disabled:
      poll(10)
      sleep(0.1)

  match radio.value:
    case "Save local model to drive":
      !rm -r "{backup}"
      !mv "{folder}" "{backup}"
    case "Keep remote model":
      !rm -r "{folder}"


if Path("/content/drive").is_mount():
  !mkdir -p "{BACKUPS_PATH}"
  !mkdir -p "{non_bak_path}"

  if not Path(LOGS_PATH).is_symlink():
    for folder in non_bak_folders:
      folder = Path(f"{LOGS_PATH}/{folder}")
      backup = Path(f"{BACKUPS_PATH}/{folder.name}")

      !mkdir -p "{folder}"
      !mv "{folder}" "{non_bak_path}" &> /dev/null
      !rm -rf "{folder}"
      folder = Path(f"{non_bak_path}/{folder.name}")
      if backup.exists() and backup.resolve() != folder.resolve():
        !rm -r "{backup}"
      !ln -s "{folder}" "{backup}" &> /dev/null

    for model in Path(LOGS_PATH).iterdir():
      if model.is_dir() and not model.is_symlink():
        backup = Path(f"{BACKUPS_PATH}/{model.name}")

        if model.name == ".ipynb_checkpoints":
          continue

        if backup.exists() and backup.is_dir():
          sync_folders(model, backup)
        else:
          !rm "{backup}"
          !mv "{model}" "{backup}"

    !rm -r "{LOGS_PATH}"
    !ln -s "{BACKUPS_PATH}" "{LOGS_PATH}"

    clear_output()
    print("✅ Models are synced!")

  else:
    !rm "{LOGS_PATH}"
    !ln -s "{BACKUPS_PATH}" "{LOGS_PATH}"
    clear_output()
    print("✅ Models already synced!")

else:
  print("❌ Drive is not mounted, skipping model syncing")
  print("To sync your models, first mount your Google Drive and re-run this cell")

In [ ]:
import os
logs_dir = "/content/Applio/logs"
for name in sorted(os.listdir(logs_dir)):
    print(name)

In [ ]:
import shutil
os.makedirs("/content/dataset_혁오_short", exist_ok=True)
shutil.copy("/content/혁오.mp3", "/content/dataset_혁오_short/")  # 확장자 실제대로
# train_jobs의 경로도 "/content/dataset_혁오_short"로 바꿔주기

In [ ]:
import os, shutil

# 혁오_short
os.makedirs("/content/dataset_혁오_short", exist_ok=True)
shutil.copy("/content/혁오.mp3", "/content/dataset_혁오_short/")

# 혁오_long
os.makedirs("/content/dataset_혁오_long", exist_ok=True)
shutil.copy("/content/혁오_long.mp3", "/content/dataset_혁오_long/")

print("✅ 복사 완료")
print("혁오_short:", os.listdir("/content/dataset_혁오_short"))
print("혁오_long:", os.listdir("/content/dataset_혁오_long"))

In [ ]:
import os
import shutil
import subprocess

ROOT = "/content/Applio"


# ===== 2) 학습 (short 먼저) =====
train_jobs = {
    "혁오_long":  "/content/dataset_혁오_long",
}

TOTAL_EPOCH = 100
SAVE_EVERY = 10
BATCH_SIZE = 8

def run_step(args, step_name, model_name):
    result = subprocess.run(args, cwd=ROOT, capture_output=True, text=True)
    print(f"--- {step_name} STDOUT ---")
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print(f"--- {step_name} STDERR ---")
        print(result.stderr[-2000:])
        print(f"❌ {model_name} {step_name} 실패! 여기서 멈춤")
        return False
    print(f"✅ {model_name} {step_name} 성공")
    return True

for model_name, dataset_path in train_jobs.items():
    print(f"\n{'='*40}\n🔄 {model_name} 학습 시작 (dataset: {dataset_path})\n{'='*40}")

    if not os.path.exists(dataset_path):
        print(f"❌ dataset_path 없음: {dataset_path}"); continue

    print(f"📂 dataset_path 내용: {os.listdir(dataset_path)}")

    ok = run_step([
        "python", "core.py", "preprocess",
        "--model_name", model_name,
        "--dataset_path", dataset_path,
        "--sample_rate", "40000",
        "--cpu_cores", "2",
        "--cut_preprocess", "Automatic",
    ], "preprocess", model_name)
    if not ok: continue

    ok = run_step([
        "python", "core.py", "extract",
        "--model_name", model_name,
        "--f0_method", "rmvpe",
        "--cpu_cores", "2",
        "--gpu", "0",
        "--sample_rate", "40000",
        "--embedder_model", "contentvec",
        "--include_mutes", "2",
    ], "extract", model_name)
    if not ok: continue

    ok = run_step([
        "python", "core.py", "train",
        "--model_name", model_name,
        "--save_every_epoch", str(SAVE_EVERY),
        "--save_only_latest", "True",
        "--save_every_weights", "True",
        "--total_epoch", str(TOTAL_EPOCH),
        "--sample_rate", "40000",
        "--batch_size", str(BATCH_SIZE),
        "--gpu", "0",
        "--pretrained", "True",
        "--vocoder", "HiFi-GAN",
    ], "train", model_name)
    if not ok: continue

    print(f"🎉 {model_name} 전체 완료!")

print("\n끝!")

In [ ]:
import os, re, subprocess

ROOT = "/content/Applio"
logs_dir = f"{ROOT}/logs"
input_dir = "/content"
output_dir = "/content/outputs"
os.makedirs(output_dir, exist_ok=True)

SONGS_A = ["장마.mp3", "dry flower.mp3", "눈사람.mp3", "밤의 여왕 아리아.mp3"]
SONGS_B = SONGS_A + ["memory.mp3", "cruel summer.mp3"]

# 노래 원곡 성별
SONG_GENDER = {
    "장마.mp3": "f",
    "dry flower.mp3": "m",
    "눈사람.mp3": "m",
    "밤의 여왕 아리아.mp3": "f",
    "memory.mp3": "m",
    "cruel summer.mp3": "f",
}

def get_epoch(fname):
    m = re.search(r"_(\d+)e_", fname)
    return int(m.group(1)) if m else None

def find_index(folder):
    idx = [f for f in os.listdir(folder) if f.endswith(".index")]
    return os.path.join(folder, idx[0]) if idx else None

def find_weights(folder, wanted_epochs):
    pth_files = [f for f in os.listdir(folder) if f.endswith(".pth") and not f.startswith(("G_", "D_"))]
    if wanted_epochs is None:
        no_epoch = [(None, f) for f in pth_files if get_epoch(f) is None]
        return no_epoch if no_epoch else [(None, pth_files[0])] if pth_files else []
    return sorted([(e, f) for e, f in [(get_epoch(f), f) for f in pth_files] if e in wanted_epochs])

def resolve_folder(name_hint):
    if os.path.exists(os.path.join(logs_dir, name_hint)):
        return name_hint
    for d in os.listdir(logs_dir):
        if name_hint.lower() in d.lower():
            print(f"   ℹ️ '{name_hint}' → 자동매칭: '{d}'")
            return d
    return name_hint

def run_infer(input_path, output_path, pth_path, index_path, pitch):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    subprocess.run([
        "python", "core.py", "infer",
        "--pitch", str(pitch),
        "--index_rate", "0.3",
        "--volume_envelope", "1",
        "--protect", "0.33",
        "--f0_method", "rmvpe",
        "--input_path", input_path,
        "--output_path", output_path,
        "--pth_path", pth_path,
        "--index_path", index_path,
        "--export_format", "WAV",
        "--embedder_model", "contentvec",
    ], cwd=ROOT)

def get_pitches(model_gender, song_gender):
    """모델 성별과 노래 원곡 성별이 같으면 0만, 다르면 보정값(+12/-12)까지"""
    if model_gender is None or song_gender is None:
        return [0]
    if model_gender == song_gender:
        return [0]
    if model_gender == "m" and song_gender == "f":
        return [0, -12]   # 남성 모델이 원래 여자 노래 부를 때 → 내려서도 비교
    if model_gender == "f" and song_gender == "m":
        return [0, 12]    # 여성 모델이 원래 남자 노래 부를 때 → 올려서도 비교
    return [0]

def process_job(folder_name_hint, wanted_epochs, gender, songs, label=None):
    folder_name = resolve_folder(folder_name_hint)
    label = label or folder_name
    folder = os.path.join(logs_dir, folder_name)
    if not os.path.exists(folder):
        print(f"⚠️ {label} 폴더 없음, 스킵"); return
    index_path = find_index(folder)
    if not index_path:
        print(f"⚠️ {label} index 없음, 스킵"); return
    weights = find_weights(folder, wanted_epochs)
    if not weights:
        print(f"⚠️ {label} 원하는 epoch weight 없음, 스킵"); return

    for epoch, weight_file in weights:
        pth_path = os.path.join(folder, weight_file)
        epoch_label = f"epoch{epoch}" if epoch is not None else "zip"
        for song in songs:
            song_path = os.path.join(input_dir, song)
            if not os.path.exists(song_path):
                print(f"   ⚠️ 원곡 없음: {song}"); continue
            song_name = os.path.splitext(song)[0]
            pitches = get_pitches(gender, SONG_GENDER.get(song))
            for pitch in pitches:
                pitch_label = f"pitch{pitch:+d}" if pitch != 0 else "pitch0"
                out_name = f"{song_name}_변환_{epoch_label}_{pitch_label}.wav"
                out_path = os.path.join(output_dir, label, out_name)
                print(f"🎵 {label} / {epoch_label} / {song} / {pitch_label} 변환...")
                run_infer(song_path, out_path, pth_path, index_path, pitch)

def process_baekyerin(epoch_folder_map, gender, songs, label="백예린"):
    for epoch, folder_name in epoch_folder_map.items():
        folder = os.path.join(logs_dir, folder_name)
        if not os.path.exists(folder):
            print(f"⚠️ {label} epoch{epoch} 폴더({folder_name}) 없음, 스킵"); continue
        index_path = find_index(folder)
        if not index_path:
            print(f"⚠️ {label} epoch{epoch} index 없음, 스킵"); continue
        pth_files = [f for f in os.listdir(folder) if f.endswith(".pth") and get_epoch(f) == epoch]
        if not pth_files:
            print(f"⚠️ {label} epoch{epoch} weight 없음, 스킵"); continue
        pth_path = os.path.join(folder, pth_files[0])

        for song in songs:
            song_path = os.path.join(input_dir, song)
            if not os.path.exists(song_path):
                print(f"   ⚠️ 원곡 없음: {song}"); continue
            song_name = os.path.splitext(song)[0]
            pitches = get_pitches(gender, SONG_GENDER.get(song))
            for pitch in pitches:
                pitch_label = f"pitch{pitch:+d}" if pitch != 0 else "pitch0"
                out_name = f"{song_name}_변환_epoch{epoch}_{pitch_label}.wav"
                out_path = os.path.join(output_dir, label, out_name)
                print(f"🎵 {label} / epoch{epoch}({folder_name}) / {song} / {pitch_label} 변환...")
                run_infer(song_path, out_path, pth_path, index_path, pitch)

# ====== 그룹 A: epoch 비교 ======
process_baekyerin(
    {10: "백예린 ", 30: "백예린_30", 50: "백예린_200", 100: "백예린_200", 200: "백예린_200"},
    "f", SONGS_A, label="백예린"
)
process_job("YerinBaek", None, "f", SONGS_A, label="백예린_허깅페이스")
process_job("혁오_short", [10,50,100,200], "m", SONGS_A)
#process_job("혁오_long",  [10,50,100,200], "m", SONGS_A)

# ====== 그룹 B: 아티스트 비교 (전부 100epoch, zip은 그대로) ======
process_job("파파로티",            [100], "m", SONGS_B)
process_job("최유리",              [100], "f", SONGS_B)
process_job("명탐정 코난 한국성우", [100], "m", SONGS_B, label="명탐정코난_한국성우")
process_job("지드래곤",            [100], "m", SONGS_B)
process_job("김광석",              [100], "m", SONGS_B)
process_job("서은광",              [100], "m", SONGS_B)
process_job("혁오_long",           [100], "m", SONGS_B)
process_job("YerinBaek",           None,  "f", SONGS_B, label="백예린_허깅페이스")
process_job("yuuriOV2",            None,  "m", SONGS_B, label="유우리")
process_job("hanni",               None,  "f", SONGS_B, label="뉴진스하니")
process_job("ConanEdogawa",        None,  "m", SONGS_B, label="명탐정코난_일본성우")
process_job("michael",             None,  "m", SONGS_B, label="마이클잭슨")
process_job("AriEternal",          None,  "f", SONGS_B, label="아리아나그란데")

print("\n🎉 전체 변환 끝!")

In [ ]:
import os, re, subprocess

ROOT = "/content/Applio"
logs_dir = f"{ROOT}/logs"
input_dir = "/content"
output_dir = "/content/outputs"

# ===== 1) 인덱스 생성 (혁오_short, 파파로티) =====
for model_name in ["혁오_short", "파파로티"]:
    print(f"🔧 {model_name} index 생성 중...")
    subprocess.run([
        "python", "core.py", "index",
        "--model_name", model_name,
        "--index_algorithm", "Auto",
    ], cwd=ROOT)

In [ ]:
def get_epoch(fname):
    m = re.search(r"_(\d+)e_", fname)
    return int(m.group(1)) if m else None

def find_index(folder):
    idx = [f for f in os.listdir(folder) if f.endswith(".index")]
    return os.path.join(folder, idx[0]) if idx else None

def find_weights(folder, wanted_epochs):
    pth_files = [f for f in os.listdir(folder) if f.endswith(".pth") and not f.startswith(("G_", "D_"))]
    if wanted_epochs is None:
        no_epoch = [(None, f) for f in pth_files if get_epoch(f) is None]
        return no_epoch if no_epoch else [(None, pth_files[0])] if pth_files else []
    return sorted([(e, f) for e, f in [(get_epoch(f), f) for f in pth_files] if e in wanted_epochs])

def best_available_epoch(folder, preferred_list):
    """preferred_list 중 있는 거 우선, 없으면 그 폴더에 있는 가장 높은 epoch 사용"""
    pth_files = [f for f in os.listdir(folder) if f.endswith(".pth") and not f.startswith(("G_", "D_"))]
    epochs = sorted([get_epoch(f) for f in pth_files if get_epoch(f) is not None])
    if not epochs:
        return None
    for p in preferred_list:
        if p in epochs:
            return p
    return max(epochs)  # 없으면 제일 높은 거라도

def resolve_folder(name_hint):
    if os.path.exists(os.path.join(logs_dir, name_hint)):
        return name_hint
    for d in os.listdir(logs_dir):
        if name_hint.lower() in d.lower():
            print(f"   ℹ️ '{name_hint}' → 자동매칭: '{d}'")
            return d
    return name_hint

def run_infer(input_path, output_path, pth_path, index_path, pitch):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    subprocess.run([
        "python", "core.py", "infer",
        "--pitch", str(pitch),
        "--index_rate", "0.3",
        "--volume_envelope", "1",
        "--protect", "0.33",
        "--f0_method", "rmvpe",
        "--input_path", input_path,
        "--output_path", output_path,
        "--pth_path", pth_path,
        "--index_path", index_path,
        "--export_format", "WAV",
        "--embedder_model", "contentvec",
    ], cwd=ROOT)

def get_pitches(model_gender, song_gender):
    if model_gender is None or song_gender is None:
        return [0]
    if model_gender == song_gender:
        return [0]
    if model_gender == "m" and song_gender == "f":
        return [0, -12]
    if model_gender == "f" and song_gender == "m":
        return [0, 12]
    return [0]

SONG_GENDER = {
    "장마.mp3": "f", "dry flower.mp3": "m", "눈사람.mp3": "m",
    "밤의 여왕 아리아.mp3": "f", "memory.mp3": "m", "cruel summer.mp3": "f",
}

def process_job(folder_name_hint, wanted_epochs, gender, songs, label=None):
    folder_name = resolve_folder(folder_name_hint)
    label = label or folder_name
    folder = os.path.join(logs_dir, folder_name)
    if not os.path.exists(folder):
        print(f"⚠️ {label} 폴더 없음, 스킵"); return
    index_path = find_index(folder)
    if not index_path:
        print(f"⚠️ {label} index 없음, 스킵"); return
    weights = find_weights(folder, wanted_epochs)
    if not weights:
        print(f"⚠️ {label} 원하는 epoch weight 없음, 스킵"); return

    for epoch, weight_file in weights:
        pth_path = os.path.join(folder, weight_file)
        epoch_label = f"epoch{epoch}" if epoch is not None else "zip"
        for song in songs:
            song_path = os.path.join(input_dir, song)
            if not os.path.exists(song_path):
                print(f"   ⚠️ 원곡 없음: {song}"); continue
            song_name = os.path.splitext(song)[0]
            pitches = get_pitches(gender, SONG_GENDER.get(song))
            for pitch in pitches:
                pitch_label = f"pitch{pitch:+d}" if pitch != 0 else "pitch0"
                out_name = f"{song_name}_변환_{epoch_label}_{pitch_label}.wav"
                out_path = os.path.join(output_dir, label, out_name)
                print(f"🎵 {label} / {epoch_label} / {song} / {pitch_label} 변환...")
                run_infer(song_path, out_path, pth_path, index_path, pitch)

SONGS_A = ["장마.mp3", "dry flower.mp3", "눈사람.mp3", "밤의 여왕 아리아.mp3"]
SONGS_B = SONGS_A + ["memory.mp3", "cruel summer.mp3"]

# ===== 2) 빠진 것만 돌리기 =====

# 혁오_short: 그룹A(10/50/100), 그룹B(100) 둘 다 통째로 스킵됐었음 → 전부 다시
process_job("혁오_short", [10, 50, 100], "m", SONGS_A)
process_job("혁오_short", [100], "m", SONGS_B)

# 파파로티: 100 없으면 있는 것 중 가장 높은 epoch로 자동 대체
papa_folder = os.path.join(logs_dir, "파파로티")
papa_epoch = best_available_epoch(papa_folder, [100])
print(f"📌 파파로티 사용 epoch: {papa_epoch}")
process_job("파파로티", [papa_epoch], "m", SONGS_B)

# cruel summer만 빠졌던 나머지 아티스트들 — 이 곡만 다시 돌림
artists_needing_cruel = [
    ("최유리", [100], "f", "최유리"),
    ("명탐정 코난 한국성우", [100], "m", "명탐정코난_한국성우"),
    ("지드래곤", [100], "m", "지드래곤"),
    ("김광석", [100], "m", "김광석"),
    ("서은광", [100], "m", "서은광"),
    ("YerinBaek", None, "f", "백예린_허깅페이스"),
    ("yuuriOV2", None, "m", "유우리"),
    ("hanni", None, "f", "뉴진스하니"),
    ("ConanEdogawa", None, "m", "명탐정코난_일본성우"),
    ("michael", None, "m", "마이클잭슨"),
    ("AriEternal", None, "f", "아리아나그란데"),
]
for folder_hint, epochs, gender, label in artists_needing_cruel:
    process_job(folder_hint, epochs, gender, ["cruel summer.mp3"], label=label)

print("\n🎉 빠진 부분 전부 보완 완료!")

In [ ]:
import os, shutil, glob

base = "/content/_export"
input_dir = os.path.join(base, "1_인풋")
mid_dir = os.path.join(base, "2_중간생성물_체크포인트")
result_dir = os.path.join(base, "3_결과물_모델")
audio_result_dir = os.path.join(base, "4_결과물_변환음원")

for d in [input_dir, mid_dir, result_dir, audio_result_dir]:
    os.makedirs(d, exist_ok=True)

logs_dir = "/content/Applio/logs"

# ===== 1) 인풋 =====
for ds_folder in glob.glob("/content/dataset_*"):
    dst = os.path.join(input_dir, "datasets", os.path.basename(ds_folder))
    shutil.copytree(ds_folder, dst, dirs_exist_ok=True)

song_list = ["장마.mp3", "dry flower.mp3", "눈사람.mp3", "밤의 여왕 아리아.mp3", "memory.mp3", "cruel summer.mp3"]
songs_dst = os.path.join(input_dir, "변환원곡")
os.makedirs(songs_dst, exist_ok=True)
for song in song_list:
    src = os.path.join("/content", song)
    if os.path.exists(src):
        shutil.copy(src, songs_dst)

# ===== 2) 중간생성물: D_/G_ pth만 =====
# ===== 3) 결과물: 진짜 모델 weight(epoch pth) + index만 =====
for model_name in sorted(os.listdir(logs_dir)):
    model_path = os.path.join(logs_dir, model_name)
    if not os.path.isdir(model_path):
        continue

    for item in os.listdir(model_path):
        item_path = os.path.join(model_path, item)
        if not os.path.isfile(item_path):
            continue  # 폴더(extracted, f0, sliced_audios 등)는 전부 무시


        if item.endswith(".pth"):  # 진짜 모델 weight
            dst_folder = os.path.join(result_dir, model_name)
            os.makedirs(dst_folder, exist_ok=True)
            shutil.copy(item_path, dst_folder)
        elif item.endswith(".index"):
            dst_folder = os.path.join(result_dir, model_name)
            os.makedirs(dst_folder, exist_ok=True)
            shutil.copy(item_path, dst_folder)

    print(f"✅ {model_name} 분류 완료")

# ===== 4) 변환된 음원 =====
if os.path.exists("/content/outputs"):
    shutil.copytree("/content/outputs", audio_result_dir, dirs_exist_ok=True)

print("\n📦 분류 끝! zip으로 압축 중...")

shutil.make_archive("/content/1_인풋", "zip", input_dir)
shutil.make_archive("/content/2_중간생성물_체크포인트", "zip", mid_dir)
shutil.make_archive("/content/3_결과물_모델", "zip", result_dir)
shutil.make_archive("/content/4_결과물_변환음원", "zip", audio_result_dir)

print("\n✅ 완료! 4개 zip 생성:")
print(" - /content/1_인풋.zip")
print(" - /content/2_중간생성물_체크포인트.zip")
print(" - /content/3_결과물_모델.zip")
print(" - /content/4_결과물_변환음원.zip")

In [ ]:
import os

logs_dir = "/content/Applio/logs"
targets = [
    "백예린_200", "백예린_30", "백예린_50", "백예린 ", "YerinBaek",
    "혁오", "AriEternal_v2b_1000e", "ConanEdogawa",
    "명탐정 코난 한국성우", "hanni", "yuuriOV2", "지드래곤", "김광석"
]

for name in targets:
    path = os.path.join(logs_dir, name)
    print(f"\n📁 [{name}]", "✅존재" if os.path.exists(path) else "❌없음")
    if os.path.exists(path):
        for f in sorted(os.listdir(path)):
            if f.endswith((".pth", ".index")):
                print(f"   - {f}")

In [ ]:
import os

logs_dir = "/content/Applio/logs"
targets = ["파파로티", "최유리", "서은광", "zips", "혁오"]

for name in targets:
    path = os.path.join(logs_dir, name)
    print(f"\n📁 [{name}]", "✅존재" if os.path.exists(path) else "❌없음")
    if os.path.exists(path):
        for f in sorted(os.listdir(path)):
            print(f"   - {f}")

# 마이클잭슨 관련 이름이 logs 어디 숨어있는지 검색
print("\n🔍 'michael' 또는 '잭슨' 포함된 폴더/파일 검색:")
for root, dirs, files in os.walk(logs_dir):
    for d in dirs:
        if "michael" in d.lower() or "잭슨" in d:
            print("  폴더:", os.path.join(root, d))
    for f in files:
        if "michael" in f.lower() or "잭슨" in f:
            print("  파일:", os.path.join(root, f))

In [ ]:
import subprocess
subprocess.run([
    "python", "core.py", "download",
    "--model_link", "https://huggingface.co/mjfan1999/MichaelJacksonThriller/resolve/main/MichaelJacksonThriller.zip"
], cwd="/content/Applio")

In [ ]:
import subprocess
subprocess.run([
    "python", "core.py", "index",
    "--model_name", "파파로티",
    "--index_algorithm", "Auto",
], cwd="/content/Applio")

In [ ]:
import os, re, subprocess

ROOT = "/content/Applio"
logs_dir = f"{ROOT}/logs"
input_dir = "/content"      # 장마.wav, dry_flower.wav, 눈사람.wav, 밤의여왕아리아.wav, memory.wav, cruel_summer.wav 들어있어야 함
output_dir = "/content/outputs"
os.makedirs(output_dir, exist_ok=True)

SONGS_A = ["장마.wav", "dry_flower.wav", "눈사람.wav", "밤의여왕아리아.wav"]
SONGS_B = SONGS_A + ["memory.wav", "cruel_summer.wav"]

def get_epoch(fname):
    m = re.search(r"_(\d+)e_", fname)
    return int(m.group(1)) if m else None

def find_index(folder):
    idx = [f for f in os.listdir(folder) if f.endswith(".index")]
    return os.path.join(folder, idx[0]) if idx else None

def find_weights(folder, wanted_epochs):
    pth_files = [f for f in os.listdir(folder) if f.endswith(".pth")]
    if wanted_epochs is None:  # zip형: epoch 패턴 없는 단일 모델
        no_epoch = [(None, f) for f in pth_files if get_epoch(f) is None]
        return no_epoch if no_epoch else [(None, pth_files[0])] if pth_files else []
    return sorted([(get_epoch(f), f) for f in pth_files if get_epoch(f) in wanted_epochs])

def run_infer(input_path, output_path, pth_path, index_path, pitch):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    subprocess.run([
        "python", "core.py", "infer",
        "--pitch", str(pitch),
        "--index_rate", "0.3",
        "--volume_envelope", "1",
        "--protect", "0.33",
        "--f0_method", "rmvpe",
        "--input_path", input_path,
        "--output_path", output_path,
        "--pth_path", pth_path,
        "--index_path", index_path,
        "--export_format", "WAV",
        "--embedder_model", "contentvec",
    ], cwd=ROOT)

def process_job(folder_name, wanted_epochs, gender, songs):
    folder = os.path.join(logs_dir, folder_name)
    if not os.path.exists(folder):
        print(f"⚠️ {folder_name} 폴더 없음, 스킵"); return
    index_path = find_index(folder)
    if not index_path:
        print(f"⚠️ {folder_name} index 없음, 스킵"); return
    weights = find_weights(folder, wanted_epochs)
    if not weights:
        print(f"⚠️ {folder_name} 원하는 epoch weight 없음, 스킵"); return

    pitches = [0]
    if gender == "f": pitches.append(12)
    elif gender == "m": pitches.append(-12)
    else: print(f"   (⚠ {folder_name} gender 미확인 → pitch 0만 실행)")

    for epoch, weight_file in weights:
        pth_path = os.path.join(folder, weight_file)
        epoch_label = f"epoch{epoch}" if epoch is not None else "zip"
        for song in songs:
            song_path = os.path.join(input_dir, song)
            if not os.path.exists(song_path):
                print(f"   ⚠️ 원곡 없음: {song}"); continue
            song_name = os.path.splitext(song)[0]
            for pitch in pitches:
                pitch_label = f"pitch{pitch:+d}" if pitch != 0 else "pitch0"
                out_name = f"{song_name}_변환_{epoch_label}_{pitch_label}.wav"
                out_path = os.path.join(output_dir, folder_name, out_name)
                print(f"🎵 {folder_name} / {epoch_label} / {song} / {pitch_label} 변환...")
                run_infer(song_path, out_path, pth_path, index_path, pitch)

# ====== 그룹 A: epoch 비교 ======
process_job("백예린",     [10,30,50,100,200], "f", SONGS_A)
process_job("백예린zip",  None,                "f", SONGS_A)
process_job("혁오", [10,50,100,200],     "m", SONGS_A)
process_job("혁오_long",  [10,50,100,200],     "m", SONGS_A)

# ====== 그룹 B: 아티스트 비교 (전부 100epoch, zip은 그대로) ======
process_job("파파로티",            [100], "m",  SONGS_B)
process_job("최유리",              [100], "f",  SONGS_B)
process_job("명탐정코난_한국성우", [100], "m",  SONGS_B)
process_job("지드래곤",            [100], "m",  SONGS_B)
process_job("김광석",              [100], "m",  SONGS_B)
process_job("서은광",              [100], "m",  SONGS_B)
process_job("혁오_long",           [100], "m",  SONGS_B)
process_job("백예린zip",           None,  "f",  SONGS_B)
process_job("유우리",              None,  "m",  SONGS_B)
process_job("하니",                None,  "f",  SONGS_B)
process_job("에도가와코난",        None,  "m",  SONGS_B)
process_job("마이클잭슨",          None,  "m",  SONGS_B)
process_job("아리아나그란데",      None,  "f",  SONGS_B)

print("\n🎉 전체 변환 끝!")

In [ ]:
# @title **Start server**
# @markdown  ### Choose a sharing method:
from IPython.display import clear_output

method = "gradio"  # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = "If you selected the 'ngrok' method, obtain your auth token here: https://dashboard.ngrok.com/get-started/your-authtoken" # @param {type:"string"}
tensorboard = True #@param {type: "boolean"}

%cd /content/{repo_name}
clear_output()

if tensorboard:
  %load_ext tensorboard
  %tensorboard --logdir logs --bind_all

match method:
  case 'gradio':
    !python app.py --listen --share --client
  case 'localtunnel':
    !echo Password IP: $(curl --silent https://ipv4.icanhazip.com)
    !echo
    !lt --port 6969 & python app.py --listen --client
  case 'ngrok':
    import ngrok
    ngrok.kill()
    listener = await ngrok.forward(6969, authtoken=ngrok_token)
    print(f"Ngrok URL: {listener.url()}")
    !python app.py --listen --client